# 📊 INF01090 - Ciência de Dados - Classification Techniques

**Spaceship Titanic - Kaggle-Style Competition**

Spaceship Titanic (<https://www.kaggle.com/competitions/spaceship-titanic/>) is a popular Kaggle competition that challenges participants to predict whether a passenger was transported to an alternate dimension, using classification models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 8,693 training passengers
- **Features:** 13 explanatory variables
- **Primary goal:** Predict whether a passenger was transported


## 📂 Dataset and Objective

The dataset describes various attributes of passengers—covering their home planet, cryosleep status, cabin, age, VIP status, and amenities usage. The task is to predict the binary outcome of whether they were transported, making it a supervised classification problem.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as logistic regression, random forest, and gradient boosting classifiers
- **Evaluation:** models are ranked by **Accuracy** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a classification pipeline that predicts **`Transported`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- classification modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://lab06inf01090-puo9j5ajrfsar22xxfmeez.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
PassengerId,prediction
8141_01,0
6034_01,0
3946_01,1
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `Transported` should be True or False


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline classification model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- checking class balance for Transported
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [ ] `train_student.csv` loads correctly
- [ ] `test_student.csv` has the same predictor columns as expected
- [ ] your preprocessing works for both train and test
- [ ] your model produces one prediction per test row
- [ ] the output file has exactly two columns: `Id`, `prediction`
- [ ] all predictions are numeric
- [ ] all predictions are boolean (True/False)


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [108]:
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgbm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names",
    category=UserWarning,
)


## 1. Load the data

Update the paths if necessary.


In [109]:
train_df = pd.read_csv("train_student.csv")
test_df = pd.read_csv("test_student.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (6085, 14)
Test shape: (2608, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,3652_01,Earth,True,G/591/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,0.0,Lina Frandriquez,False
1,8449_01,Europa,False,C/317/S,55 Cancri e,32.0,False,0.0,2154.0,0.0,9655.0,859.0,Arrain Purellliss,False
2,1871_01,Europa,False,C/70/S,TRAPPIST-1e,31.0,NaN,0.0,1983.0,0.0,172.0,52.0,Aston Spedlybood,True
3,1811_01,Earth,True,G/285/S,PSO J318.5-22,25.0,False,0.0,0.0,0.0,0.0,0.0,Gerry Georgasey,True
4,8589_01,Europa,False,D/267/P,55 Cancri e,26.0,False,6.0,728.0,0.0,1273.0,1116.0,Gacrab Gealter,False


## 2. Identify target and predictors


In [110]:
target_col = "Transported"
id_col = "PassengerId"

X = train_df.drop(columns=[target_col])
y = train_df[target_col].copy()

print("Target summary:")
display(y.describe())


Target summary:


count     6085
unique       2
top       True
freq      3065
Name: Transported, dtype: object

### 2.1 Missing-value handling

In [111]:
# Null values analysis
numeric_missing = X.select_dtypes(include=["number"]).isnull().sum()
print("Numeric features with missing values:")
print(numeric_missing[numeric_missing > 0])

categorical_missing = X.select_dtypes(exclude=["number"]).isnull().sum()
print("Categorical features with missing values:")
print(categorical_missing[categorical_missing > 0])

print("duplicated rows:", X.duplicated().sum())

print("Number of VIPS that are in Cryo:", X[(X["VIP"] == True) & (X["CryoSleep"] == True)].shape[0])

Numeric features with missing values:
Age             123
RoomService     129
FoodCourt       134
ShoppingMall    133
Spa             128
VRDeck          133
dtype: int64
Categorical features with missing values:
HomePlanet     149
CryoSleep      147
Cabin          138
Destination    139
VIP            144
Name           144
dtype: int64
duplicated rows: 0
Number of VIPS that are in Cryo: 14


### 2.2 Checking outliers

In [112]:
#somehow, check the outliers

### 2.3 Feature Encoding

In [113]:
def add_engineered_features(df):
    df = df.copy()

    expense_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    expense_data = df[expense_cols].fillna(0)

    # Spending behavior is one of the strongest signals in this problem.
    df["Missing_Expense_Count"] = df[expense_cols].isna().sum(axis=1)
    df["Total_Spent"] = expense_data.sum(axis=1)
    df["Luxury_Spend"] = expense_data[["RoomService", "Spa", "VRDeck"]].sum(axis=1)
    df["Social_Spend"] = expense_data[["FoodCourt", "ShoppingMall"]].sum(axis=1)
    df["Has_Spent"] = (df["Total_Spent"] > 0).astype(int)
    df["Luxury_Spend_Ratio"] = np.where(
        df["Total_Spent"] > 0,
        df["Luxury_Spend"] / df["Total_Spent"],
        0,
    )
    df["Social_Spend_Ratio"] = np.where(
        df["Total_Spent"] > 0,
        df["Social_Spend"] / df["Total_Spent"],
        0,
    )
    for col in expense_cols:
        df[f"Log_{col}"] = np.log1p(expense_data[col])
    df["Log_Total_Spent"] = np.log1p(df["Total_Spent"])
    df["Log_Luxury_Spend"] = np.log1p(df["Luxury_Spend"])
    df["Log_Social_Spend"] = np.log1p(df["Social_Spend"])

    spend_conditions = [
        df["Total_Spent"].eq(0),
        df["Luxury_Spend"] > df["Social_Spend"],
        df["Social_Spend"] > df["Luxury_Spend"],
    ]
    df["Spending_Profile"] = np.select(
        spend_conditions,
        ["NoSpend", "LuxuryHeavy", "SocialHeavy"],
        default="Mixed",
    )

    passenger_parts = df["PassengerId"].str.split("_", expand=True)
    df["Passenger_Group"] = passenger_parts[0]
    df["Passenger_Number"] = pd.to_numeric(passenger_parts[1], errors="coerce")
    df["Group_Size"] = df.groupby("Passenger_Group")["PassengerId"].transform("count")
    df["Is_Alone"] = (df["Group_Size"] == 1).astype(int)

    df["Name_Missing"] = df["Name"].isna().astype(int)
    df["Last_Name"] = df["Name"].str.split().str[-1]
    df["Family_Size"] = df.groupby("Last_Name")["PassengerId"].transform("count")
    df["Family_Size"] = df["Family_Size"].fillna(0)
    df["Has_Family_Onboard"] = (df["Family_Size"] > 1).astype(int)

    df["Cabin_Missing"] = df["Cabin"].isna().astype(int)
    cabin_parts = df["Cabin"].str.split("/", expand=True)
    df["Passenger_Deck"] = cabin_parts[0]
    df["Passenger_Num"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    df["Passenger_Side"] = cabin_parts[2]
    df["Cabin_Region"] = pd.cut(
        df["Passenger_Num"],
        bins=[-1, 300, 600, 900, 1200, 1500, 1800, np.inf],
        labels=["0-300", "301-600", "601-900", "901-1200", "1201-1500", "1501-1800", "1801+"],
    )
    df["Deck_Side"] = df["Passenger_Deck"].fillna("Missing") + "_" + df["Passenger_Side"].fillna("Missing")

    df["Age_Missing"] = df["Age"].isna().astype(int)
    age_for_ratio = df["Age"].fillna(df["Age"].median())
    df["Age_Group"] = pd.cut(
        df["Age"],
        bins=[-1, 12, 18, 35, 55, np.inf],
        labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"],
    )
    df["Total_Spent_per_Age"] = df["Total_Spent"] / (age_for_ratio + 1)

    df["Travel_Route"] = df["HomePlanet"].fillna("Missing") + "_to_" + df["Destination"].fillna("Missing")
    df["HomePlanet_Deck"] = df["HomePlanet"].fillna("Missing") + "_" + df["Passenger_Deck"].fillna("Missing")
    df["Cryo_Destination"] = df["CryoSleep"].astype(str) + "_" + df["Destination"].fillna("Missing")
    df["Cryo_No_Spend"] = ((df["CryoSleep"] == True) & (df["Has_Spent"] == 0)).astype(int)
    df["Awake_No_Spend"] = ((df["CryoSleep"] == False) & (df["Has_Spent"] == 0)).astype(int)
    df["VIP_Spender"] = ((df["VIP"] == True) & (df["Total_Spent"] > 0)).astype(int)
    df["Is_Rich"] = ((df["VIP"] == True) | (df["Total_Spent"] > 2000)).astype(int)

    drop_columns = ["PassengerId", "Name", "Cabin", "Last_Name", "Passenger_Group"]

    return df.drop(columns=drop_columns, errors="ignore")

X = add_engineered_features(X)

print("New features added. Current shape:", X.shape)
print("New features summary:")
display(X.describe())

New features added. Current shape: (6085, 48)
New features summary:


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Missing_Expense_Count,Total_Spent,Luxury_Spend,Social_Spend,...,Family_Size,Has_Family_Onboard,Cabin_Missing,Passenger_Num,Age_Missing,Total_Spent_per_Age,Cryo_No_Spend,Awake_No_Spend,VIP_Spender,Is_Rich
count,5962.000000,5956.000000,5951.000000,5952.000000,5957.000000,5952.000000,6085.000000,6085.000000,6085.000000,6085.000000,...,6085.000000,6085.000000,6085.000000,5947.000000,6085.000000,6085.000000,6085.000000,6085.000000,6085.000000,6085.000000
mean,28.649446,228.811954,458.420938,171.714718,305.697667,298.110887,0.107970,1431.110929,814.823500,616.287428,...,4.006409,0.889400,0.022679,596.280478,0.020214,46.123503,0.348562,0.062942,0.020049,0.202958
std,14.468341,676.727091,1625.602344,555.327890,1131.157324,1111.173524,0.320273,2754.546608,1803.521575,1688.477457,...,2.303941,0.313662,0.148889,511.766665,0.140742,86.869546,0.476554,0.242878,0.140180,0.402235
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.000000,1.000000,0.000000,165.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,716.000000,58.000000,1.000000,...,4.000000,1.000000,0.000000,427.000000,0.000000,21.500000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,76.000000,23.250000,53.000000,43.000000,0.000000,1453.000000,852.000000,600.000000,...,5.000000,1.000000,0.000000,984.000000,0.000000,50.129032,1.000000,0.000000,0.000000,0.000000
max,79.000000,14327.000000,29813.000000,12253.000000,22408.000000,20336.000000,2.000000,35987.000000,28600.000000,29813.000000,...,13.000000,1.000000,1.000000,1893.000000,1.000000,1604.105263,1.000000,1.000000,1.000000,1.000000


## 3. Build a local validation split

Use this split to compare models **before** submitting to the leaderboard.


In [114]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_valid.shape)


(4868, 48) (1217, 48)


## 4. Separate numeric and categorical columns


In [115]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Numeric features: 35
Categorical features: 13


## 5. Create a preprocessing pipeline

You may improve this pipeline as part of the assignment.


In [116]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 6. Baseline model

Start with a simple model.


In [117]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)
pred_valid_baseline = baseline_model.predict(X_valid)

acc_baseline = accuracy_score(y_valid, pred_valid_baseline)
f1_baseline = f1_score(y_valid, pred_valid_baseline, average='macro')


print("Baseline Accuracy:", acc_baseline)
print("Baseline F1-Score:", f1_baseline)



Baseline Accuracy: 0.790468364831553
Baseline F1-Score: 0.7903726865408505


## 7. Improved model

Try at least one stronger model and compare the result.


In [118]:


improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=300,
        random_state=42
    ))
])

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.1,
        objective='binary:logistic'
    ))
])

improved_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=300,       # Equivalent to max_iter / number of trees
        learning_rate=0.1,      # Step size shrinkage
        max_depth=3,            # Maximum tree depth
        random_state=42,        # Random seed for reproducibility
        verbosity=-1            # Suppresses unnecessary warning messages
    ))
])

#improved_model = Pipeline(steps=[
#    ("preprocessor", preprocessor),
#    ("model", HistGradientBoostingClassifier(
#        max_iter=100,
#        learning_rate=0.1,
#        max_depth=5,
#        random_state=42,
#        categorical_features="from_dtype"
#    ))
#])

improved_model.fit(X_train, y_train)
pred_valid_improved = improved_model.predict(X_valid)

acc_improved = accuracy_score(y_valid, pred_valid_improved)
f1_improved = f1_score(y_valid, pred_valid_improved, average='macro')


print("Improved Accuracy:", acc_improved)
print("Improved F1-Score:", f1_improved)



Improved Accuracy: 0.8101889893179951
Improved F1-Score: 0.8101843755760261


## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [119]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "Accuracy": [acc_baseline, acc_improved],
    "F1-Score": [f1_baseline, f1_improved],

})

comparison


,Model,Accuracy,F1-Score
0,Baseline,0.790468,0.790373
1,Improved,0.810189,0.810184


**Write a short discussion here.**

- Which model performed better?
- Was the improvement large or small?
- What might explain the difference?


## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [120]:
final_model = improved_model  # change if needed 

final_model.fit(X, y)
test_ids = test_df[id_col].copy()
test_df = add_engineered_features(test_df)

missing_cols = [col for col in X.columns if col not in test_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in test data after preprocessing: {missing_cols}")

test_df = test_df[X.columns]

test_predictions = final_model.predict(test_df).astype(int)

submission = pd.DataFrame({
    id_col: test_ids,
    "prediction": test_predictions
})

print("Submission shape:", submission.shape)
print("Any missing predictions:", submission["prediction"].isna().any())
print("Any non-binary predictions:", (~submission["prediction"].isin([0, 1])).any())
submission.head()


Submission shape: (2608, 2)
Any missing predictions: False
Any non-binary predictions: False


,PassengerId,prediction
0,8137_01,0
1,7191_01,0
2,6837_01,0
3,4453_01,1
4,4400_03,1


## 10. Save the submission file


In [121]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://(Enter your Streamlit app URL here)/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission:
- Best submission:
- Final submitted model:


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection
